<a href="https://colab.research.google.com/github/AdrianDVnqn/UA_MDM_Labo2_Grupo12/blob/LightGBM_Test/tutoriales/05_Resnet50_2_integrar_12052025.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
####IMPORTANTE CARGAR UTILS.PY
from google.colab import files
uploaded = files.upload()

Saving utils.py to utils.py


In [2]:
!pip install optuna
!pip install -U kaleido
!pip install optuna-dashboard
#!pip install kaleido

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 386.6/386.6 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.9/231.9 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 MB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 49.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.2/104.2 kB 11.5 MB/s eta 0:00:00


In [3]:
import numpy as np
import pandas as pd

from sklearn.metrics import cohen_kappa_score, accuracy_score,balanced_accuracy_score

from plotly import express as px

from utils import plot_confusion_matrix, get_artifact_filename

import os

from json import loads

from joblib import load, dump

import optuna
from optuna.artifacts import FileSystemArtifactStore, upload_artifact

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [8]:
# Definir la ruta base para tu Google Drive
BASE_DIR_DRIVE = '/content/drive/MyDrive'
PATH_TO_TRAIN = os.path.join(BASE_DIR_DRIVE, "Colab Notebooks/LABO_II/input/petfinder-adoption-prediction/train/train_final_thres.csv")
# Artefactos a subir a optuna
PATH_TO_TEMP_FILES = os.path.join(BASE_DIR_DRIVE, "Colab Notebooks/LABO_II/work/optuna_temp_artifacts")
# Artefactos que optuna gestiona
PATH_TO_OPTUNA_ARTIFACTS = os.path.join(BASE_DIR_DRIVE, "Colab Notebooks/LABO_II/work/optuna_artifacts")
PATH_TO_MODELS = os.path.join(BASE_DIR_DRIVE, "work/models")
PATH_TO_IMAGES_DIR = os.path.join(BASE_DIR_DRIVE, "input/petfinder-adoption-prediction/train_images")

In [6]:
# # Paths
# BASE_DIR = './'  # ya que estás en Colab
# PATH_TO_MODELS = os.path.join(BASE_DIR, "work/models")
# PATH_TO_TRAIN = os.path.join(BASE_DIR, "input/petfinder-adoption-prediction/train/train_final_thres.csv")
# PATH_TO_IMAGES_DIR = os.path.join(BASE_DIR, "input/petfinder-adoption-prediction/train_images")
# PATH_TO_TEMP_FILES = os.path.join(BASE_DIR, "work/optuna_temp_artifacts")
# PATH_TO_OPTUNA_ARTIFACTS = os.path.join(BASE_DIR, "work/optuna_artifacts")


In [9]:
# study_lgb = optuna.create_study(direction='maximize',
#                             storage="sqlite:///../work/db.sqlite3",  # Specify the storage URL here.
#                             study_name="04 - LGB Multiclass CV",
#                             load_if_exists = True)

ruta_carpeta_work = '/content/drive/MyDrive/Colab Notebooks/LABO_II/work'

# Construye la ruta completa al archivo db.sqlite3
ruta_completa_db = os.path.join(ruta_carpeta_work, 'db_cv_12052025_1.sqlite3')

# Ahora utiliza esta ruta en la configuración de Optuna
storage = f"sqlite:///{ruta_completa_db}"

study_lgb = optuna.create_study(
    direction='maximize',
    storage=storage,
    study_name="04 - LGB Multiclass CV 12052025",
    load_if_exists=True
)

lgb_dataset = load(os.path.join(PATH_TO_OPTUNA_ARTIFACTS,get_artifact_filename(study_lgb,'test')))
#lgb_dataset = load(os.path.join('/content/drive/MyDrive/Colab Notebooks/LABO_II/work', get_artifact_filename(study_lgb,'test')))

[I 2025-05-12 21:38:44,465] Using an existing study with name '04 - LGB Multiclass CV 12052025' instead of creating a new one.


In [10]:
lgb_dataset

,Type,Name,Age,Breed1,Breed2,Gender,Color1,Color2,Color3,MaturitySize,...,stopwords_eliminadas,descripcion_para_analisis,Nombres_limpios,categoria_rescatista,cantidad_animales,disponibilidad_imagen,estado_sanitario,AgeCategory,State_importance,pred
10671,2,Elsa,2,265,0,2,1,4,7,2,...,"['is', 'is', 'this', 'is', 'for', 'me', 'to', ...","['elsa', 'female', 'kitten', 'age', 'months', ...",elsa,4,1,3,5,1,1,"[0.02322141249599711, 3.047856840081007, 1.442..."
9197,1,Gina,2,307,0,2,1,0,0,2,...,"['to', 'to', 'get', 'them', 'as', 'can', 'keep...","['pet', 'dog', 'gave', 'birth', 'puppies', 'lo...",gina,2,1,3,4,1,2,"[0.06568777179422862, 1.702299049664471, 1.152..."
7212,2,Bee,3,299,0,1,2,6,0,1,...,"['we', 'him', 'in', 'his', 'was', 'so', 'we', ...","['rescued', 'cat', 'saved', 'restaurant', 'leg...",bee,4,1,3,4,1,2,"[0.03364599028148262, 2.427540585708382, 1.487..."
10712,2,Mega,84,266,0,1,1,2,7,3,...,"['after', 'his', 'will', 'and', 'by', 'of', 'f...","['mega', 'named', 'strong', 'maggot', 'wound',...",mega,4,1,3,4,3,1,"[0.04714299718935783, 0.3655002929450484, 1.32..."
3681,2,NaN,2,265,266,3,1,2,7,1,...,['so'],"['hye', 'saya', 'ada', 'dua', 'ekor', 'anak', ...",NaN,1,2,3,5,1,3,"[0.13124458239716671, 1.9304072933891037, 1.20..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1015,1,Murni,24,5,307,2,3,5,0,2,...,"['is', 'but', 'can', 'be', 'around', 'she', 'h...","['murni', 'happy', 'doggie', 'shy', 'new', 'pe...",murni,4,1,3,1,2,1,"[0.0475220739067715, 0.4291818748300964, 0.982..."
9623,1,JoJo,72,83,0,1,1,0,0,2,...,"['was', 'by', 'he', 'was', 'an', 'and', 'he', ...","['jo', 'jo', 'black', 'cocker', 'abandoned', '...",jojo,4,1,3,1,3,1,"[0.047672862251968526, 1.3337219900503174, 1.2..."
8161,1,Little Blackie 1,1,307,0,1,1,0,0,2,...,['for'],"['puppy', 'adoption']",little blackie 1,4,1,3,5,1,1,"[1.1016573082387753, 0.8758756856520922, 1.850..."
6202,2,Baby,7,265,0,2,7,0,0,2,...,"['this', 'for', 'me', 'if', 'it', 'will', 'be'...","['hi', 'giving', 'cat', 'adoption', 'contact',...",baby,1,1,3,1,1,1,"[0.07487105849665703, 1.0594256145015262, 1.08..."


In [11]:
MODEL_NAME = '04 ResNet_1005'
MODEL_VERSION = '1.0.1'

# study_resnet = optuna.create_study(direction='maximize',
#                             storage="sqlite:///../work/db.sqlite3",  # Specify the storage URL here.
#                             study_name=f'{MODEL_NAME}_{MODEL_VERSION}',
#                             load_if_exists = True)

# Define la ruta a la carpeta 'work' en tu Google Drive
ruta_carpeta_work_resnet = '/content/drive/MyDrive/Colab Notebooks/LABO_II/work'
# Construye la ruta completa al archivo db.sqlite3
ruta_completa_db_resnet = os.path.join(ruta_carpeta_work_resnet, 'db_10052025.sqlite3')
# Ahora utiliza esta ruta en la configuración de Optuna
storage_resnet = f"sqlite:///{ruta_completa_db_resnet}"


study_resnet = optuna.create_study(
    direction='maximize',
    storage=storage_resnet,
    study_name=f'{MODEL_NAME}_{MODEL_VERSION}',
    load_if_exists=True
)


resnet_dataset = load(os.path.join(PATH_TO_OPTUNA_ARTIFACTS,get_artifact_filename(study_resnet,'test')))

[I 2025-05-12 21:38:58,685] Using an existing study with name '04 ResNet_1005_1.0.1' instead of creating a new one.


In [12]:
resnet_dataset

,PetID,pred,Type,Name,Age,Breed1,Breed2,Gender,Color1,Color2,...,cantidad_palabras_no_stopwords,stopwords_eliminadas,descripcion_para_analisis,Nombres_limpios,categoria_rescatista,cantidad_animales,disponibilidad_imagen,estado_sanitario,AgeCategory,State_importance
0,0a4b42306,"[-1.4271206, 0.98588306, 0.37117553, 0.5535631...",1,Marley & Kiara (URGENT ADOPTION),30,109,0,3,3,0,...,26,"['we', 'are', 'for', 'for', 'our', 'and', 'as'...","['looking', 'new', 'home', 'lovely', 'golden',...",marley kiara urgent adoption,1,2,5,1,2,1
1,0a555d688,"[-1.925249, 0.039305426, 0.55242705, 0.4384596...",1,Prime,12,103,206,1,1,7,...,10,['for'],"['years', 'old', 'husky', 'mix', 'german', 'sh...",prime,1,1,3,1,2,1
2,0d6cb5b69,"[-1.8390235, 0.6105355, 0.9812042, 0.8175106, ...",1,Browny,3,307,0,1,2,0,...,23,"['anyone', 'in', 'this', 'found', 'him', 'my',...","['interested', 'adopting', 'puppy', 'near', 'h...",browny,1,1,3,9,1,1
3,12380b365,"[-1.1590437, 1.2966346, 0.37332928, -0.0472715...",1,Mun Mun,60,179,0,2,5,7,...,38,"['this', 'since', 'she', 'was', 'about', 'with...","['fostered', 'pretty', 'girl', 'year', 'old', ...",mun mun,1,1,3,1,3,1
4,13485ebf1,"[-1.6415061, 0.73820513, 0.9116153, 0.5669716,...",1,Jimmy,15,109,0,2,3,0,...,5,"['is', 'very', 'and', 'for']","['jimmy', 'playfull', 'looking', 'loving', 'ho...",jimmy,1,1,3,5,2,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2333,fe056693f,"[-1.7588289, -0.45278412, 0.58934224, -0.05705...",2,Hualulu,5,294,0,2,1,4,...,16,"['of', 'but', 'once', 'she', 'to', 'you', 'and...","['scare', 'strangers', 'gets', 'know', 'close'...",hualulu,2,1,3,1,1,1
2334,fe1c36fb1,"[-0.98359704, -0.017351912, 0.3284076, 0.09681...",2,Abu,12,292,0,2,2,5,...,62,"['she', 'was', 'but', 'have', 'been', 'her', '...","['stray', 'cat', 'feeding', 'months', 'gave', ...",abu,1,1,3,5,2,1
2335,fe41e15d5,"[-1.2830864, 0.21179813, 0.2864154, -0.0609537...",2,Cat For Adoption,7,249,257,3,1,2,...,30,"['for', 'these', 'are', 'by', 'my', 'and', 'my...","['cats', 'adoption', 'cats', 'abandoned', 'nei...",cat for adoption,1,3,3,5,1,1
2336,fe790ba60,"[-3.1831048, -0.8782205, 0.08329977, 0.8899268...",1,Whitsy,4,307,0,1,7,0,...,3,"['and', 'being', 'in']","['rescued', 'fostered', 'klang']",whitsy,4,1,3,1,1,2


In [13]:
merged_datasets = lgb_dataset[['PetID', 'pred', 'AdoptionSpeed']].rename({'pred':'lgb_pred_score'},axis=1).merge(resnet_dataset[['PetID', 'pred']].rename({'pred':'resnet_pred_score'},axis=1),
                  on='PetID', how='outer')



merged_datasets['resnet_pred_score'] = [np.zeros(5) if type(i) is float else  i for i in merged_datasets['resnet_pred_score'] ]

In [14]:
merged_datasets['resnet_pred_score']

,resnet_pred_score
0,"[-1.9458486, -0.0043535884, 1.2023433, 1.10391..."
1,"[-2.1744244, -0.2983307, 1.0597225, 0.62469965..."
2,"[-2.0341647, 0.84190595, 0.79452723, 0.3987306..."
3,"[-2.6787448, 1.0057539, 1.159302, 0.6220832, -..."
4,"[-1.6690625, 0.066984616, 0.37796998, 0.724677..."
...,...
2394,"[-1.4319173, 0.83807886, 1.2430862, 0.3130001,..."
2395,"[-1.6127261, 1.4383335, 1.3621353, 0.031383872..."
2396,"[-2.137112, 0.40426302, 1.129489, 0.54325914, ..."
2397,"[-1.7013699, 0.6615675, 0.500705, 0.1692328, 0..."


In [15]:
merged_datasets['blend_pred_score'] = [r['lgb_pred_score']+r['resnet_pred_score'] for i,r in merged_datasets.iterrows()]

In [16]:
merged_datasets['lgb_pred'] = [r.argmax() for r in merged_datasets['lgb_pred_score']]
merged_datasets['resnet_pred'] = [r.argmax() for r in merged_datasets['resnet_pred_score']]
merged_datasets['blended_pred'] = [r.argmax() for r in merged_datasets['blend_pred_score']]

In [17]:
plot_confusion_matrix(merged_datasets['AdoptionSpeed'],
                      merged_datasets['lgb_pred'],
                    title = 'LGB Model Kappa: ' + str(cohen_kappa_score(merged_datasets['AdoptionSpeed'],
                                                                    merged_datasets['lgb_pred'],
                                                                    weights='quadratic')))

In [18]:
plot_confusion_matrix(merged_datasets['AdoptionSpeed'],
                      merged_datasets['resnet_pred'],
                    title = 'Resnet Model Kappa: ' + str(cohen_kappa_score(merged_datasets['AdoptionSpeed'],
                                                                    merged_datasets['resnet_pred'],
                                                                    weights='quadratic')))



In [19]:
plot_confusion_matrix(merged_datasets['AdoptionSpeed'],
                      merged_datasets['blended_pred'],
                    title = 'Blended Model Kappa: ' + str(cohen_kappa_score(merged_datasets['AdoptionSpeed'],
                                                                    merged_datasets['blended_pred'],
                                                                    weights='quadratic')))
